# Company Beta Types - Historical

This article and sample code demonstrate how to use the LSEG Data Library for Python to calculate Normal Beta, Beta Up, and Beta Down for any company and index on any given date, following the methodology outlined in the __Company Beta Types – Historical Workspace Excel__ template. 

![](https://raw.githubusercontent.com/LSEG-API-Samples/Article.LDLib.Python.CompanyBeta/9cecc6bef47816aa84028a18ea4c16d0d98a8a19/company_beta_template.png)

This methodology incorporates different parameters for each type of Beta, including the calculation range and the periodicity of the data.
- __Normal Beta__ measures a security’s volatility relative to the overall market.
- __Beta Up__ measures a security’s volatility relative to the market, but only on days when the benchmark’s return is positive.
- __Beta Down__ measures a security’s volatility relative to the market, but only on days when the benchmark’s return is negative.



|<h3>Normal Beta</h3>|<h3>Beta Up</h3>|<h3>Beta Down</h3>|
|---|---|---|
|A measure of the stock price volatility relative to the benchmark market. It is a covariance of the security’s price movement in relation to the market’s price movement. Beta is calculated using the least squares linear regression formula.|A measure of the stock price volatility relative to the benchmark market. It is a covariance of the security’s price movement in relation to the market’s price movement. Beta Up is calculated using the least squares linear regression formula and only uses positive price points.|A measure of the stock price volatility relative to the benchmark market. It is a covariance of the security’s price movement in relation to the market’s price movement. Beta Down is calculated using the least squares linear regression formula and only uses negative price points.|
||An Up data point is defined as a period when the benchmark is up.|A Down data point is defined as a period when the benchmark is down.|
|Benchmark is set based on the primary exchange the security trade.|Benchmark is set based on the primary exchange the security trade.|Benchmark is set based on the primary exchange the security trade.|
|__Available periods:__<ul><li>90-days: 90 days closing price data points</li><li>180-days: 180 days closing price data points</li><li>2-year: 104 weekly closing price data points</li><li>3-year: 156 weekly closing price data points</li><li>5-year: 60 monthly closing price data points</li></ul>|__Available periods:__<ul><li>2-year: 104 weekly closing price data points</li><li>3-year: 156 weekly closing price data points</li><li>5-year: 60 monthly closing price data points</li></ul>|__Available periods:__<ul><li>2-year: 104 weekly closing price data points</li><li>3-year: 156 weekly closing price data points</li><li>5-year: 60 monthly closing price data points</li></ul>|
|Calculation of Beta requires a certain number of data points to be available. A data point consists of a percentage change for the stock and the benchmark, for the same time period. Time periods are non-overlapping, and the weekly and monthly sample sets roll ahead at the beginning of each new week or month.|Calculation of Beta requires a certain number of data points to be available. A data point consists of a percentage change for the stock and the benchmark, for the same time period. Time periods are non-overlapping, and the weekly and monthly sample sets roll ahead at the beginning of each new week or month.|Calculation of Beta requires a certain number of data points to be available. A data point consists of a percentage change for the stock and the benchmark, for the same time period. Time periods are non-overlapping, and the weekly and monthly sample sets roll ahead at the beginning of each new week or month.|
|Calculation of Beta requires at least 2/3 of non-null data points to be present.<ul><li>5 year monthly calculation require 40 or more non-null data points to be present out of 60 months total.</li><li>3 year weekly calculation require 104 or more non-null data points to be present out of 156 weeks total.</li><li>2 year weekly calculation require 70 or more non-null data points to be present out of 104 weeks total.</li><li>90-days calculation require complete 90 days of non-null data points.</li><li>180-days calculation require complete 180 days of non-null data points.</li></ul>NULL/empty value will appear if minimum data points requirement are not met.|Up calculation of Beta require at least 1/6 of data points to be present.<ul><li>5 year monthly up calculation require 10 or more non-null up months’ data points to be present.</li><li>3 year weekly up calculation require 26 or more non-null up weeks’ data points to be present.</li><li>2 year weekly up calculation require 17 or more non-null up weeks’ data points to be present.</li></ul>NULL/empty value will appear if minimum data points requirement are not met.|Down calculation of Beta require at least 1/6 of data points to be present.<ul><li>5 year monthly up calculation require 10 or more non-null up months’ data points to be present.</li><li>3 year weekly up calculation require 26 or more non-null up weeks’ data points to be present.</li><li>2 year weekly up calculation require 17 or more non-null up weeks’ data points to be present.</li></ul>NULL/empty value will appear if minimum data points requirement are not met.|
|__Normal Beta Formula:__<br>![Normal Beta Formula](normal_beta.png)|__Beta Up Formula:__<br>![Beta Up Formula](beta_up.jpg)|__Beta Down Formula:__<br>![Beta_Down Formula](beta_down.jpg)|

## Required Inputs

- __Company Code (RIC)__: Ticker (RIC) of the subject company.
- __Index Code (RIC)__: Ticker (RIC) of the subject index.
- __Historical Date__: The end-date for querying events.

## Calculation

The following steps are used to calculate the __Company Beta Types - Historical__. 

|Steps|Descriptions|
|---|----|
|Step 1|Price series of the Stock and Index|
|Step 2|Calculation of return from the stock and index|
|Step 3|Exclusion of values that are not negative or positive from the index return series|
|Step 4|Series of negative or positive Index Return|
|Step 5|Calculation of BETA's value|

## Python Code

The sample code imports and uses the following Python libraries:

- __lseg.data__: Provides high‑level APIs for accessing market data—such as prices, fundamentals, and news—from LSEG platforms (for example, the Data Platform and Workspace).
- __lseg.data.content.historical_pricing__: Exposes the historical pricing endpoint, which enables access to time‑series price data (such as OHLC, closing prices, and volume) for instruments identified by symbols or identifiers like RICs.
- __NumPy__: Supports numerical computing with arrays and vectorized mathematical operations.
- __Pandas__: Provides Series and DataFrame structures for working with tabular and time‑series data, making it well suited for pricing data analysis, resampling, joins, and rolling calculations.

Then, the code calls the __ld.open_session()__ method to establish a connection to the Workspace desktop session.


In [ ]:
import lseg.data as ld
from lseg.data.content import historical_pricing
import numpy as np
import pandas as pd
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

pd.set_option('future.no_silent_downcasting', True)

ld.open_session()


## Step 1: Price series of the Stock and Index

The function defined for this step accepts the following parameters:
- __ric__ (str): The stock RIC.
- __index__ (str): The index RIC.
- __interval__ (str): An interval string for filtering historical pricing events, such as P1D (daily), P1W (weekly), and P1M (monthly)
- __end__ (str): The end-date string for querying events, formatted as YYYY‑MM‑DD.
- __datapoints__ (int): A maximum number of rows to return

It will return a data frame.

This function uses the __historical_pricing__ endpoint to retrieve historical close pricings for both the stock and the index. It then merges the results into a single DataFrame and returns it to the caller.


In [ ]:
def Step1(ric: str, index: str, interval: str, end: str, datapoints: int) -> pd.DataFrame:
    stock_resp = historical_pricing.summaries.Definition(
        ric, 
        interval=interval,    
        end = end, 
        count = datapoints,
        fields=['TRDPRC_1'],
        extended_params={"includeTradeIndicator":"True"}
        ).get_data()
    stock_df = stock_resp.data.df.ffill()
    if 'TRADE_IND' in stock_df.columns:
        stock_df = stock_df.drop(['TRADE_IND'], axis=1)
    stock_reverse = stock_df.iloc[::-1]
    
    #Index
    index_resp = historical_pricing.summaries.Definition(
        index, 
        interval=interval,    
        end = end, 
        count = datapoints,
        fields=['TRDPRC_1'],
        extended_params={"includeTradeIndicator":"True"}
    ).get_data()
    index_df = index_resp.data.df.ffill()
    if 'TRADE_IND' in index_df.columns:
            index_df = index_df.drop(['TRADE_IND'], axis=1)
    index_reverse = index_df.iloc[::-1]
    
    df = stock_reverse.copy()
    df.columns = ['Stock Trade Close']
    df['Index Trade Close'] = index_reverse['TRDPRC_1']
    
    return df    

###  Example
The function can be called like this:

```
Step1_df = Step1('PTT.BK','.SETI', 'P1D', '2026-01-23', 91)
```

In [ ]:
Step1_df = Step1('PTT.BK','.SETI', 'P1D', '2026-01-23', 91)
Step1_df

## Step 2: Calculation of return from the stock and index

The function defined for this step takes the data frame produced in the first step as input. It uses the close prices from that data frame to calculate the returns for both the stock and the index, and then returns a new data frame to the caller.



In [ ]:
def Step2(input_df: pd.DataFrame) -> pd.DataFrame:
    #Calculate Stock Return
    stock_df = pd.DataFrame(input_df['Stock Trade Close'])[::-1]
    stock_reverse1 = stock_df.shift()[::-1]
    stock1 = np.log(stock_df / stock_reverse1)
    stock1 = stock1.dropna()
    step2 = stock1.copy()
    step2.columns = ['Stock Return']    
    
    #Calculate Index Return
    index_df = pd.DataFrame(input_df['Index Trade Close'])[::-1]
    index_reverse1 = index_df.shift()[::-1]
    index1 = np.log(index_df / index_reverse1)
    index1 = index1.dropna()
    
    step2['Index Return']=index1['Index Trade Close']
    return step2[::-1]

###  Example
The function can be called like this:

```
Step2_df = Step2(Step1_df)
```

In [ ]:
Step2_df = Step2(Step1_df)
Step2_df

## Step 3: Exclusion of values that are not negative or positive from the index return series


The function defined for this step accepts the data frame produced in the second step, along with the calculation method to be applied. 

- __input_df__ (pd.DataFrame): The data frame generated from the second step.
- __beta__ (str): The calculation method used to compute the beta value. Valid options include '', 'up', or 'down'.

If the calculation method is 'up', all zero and negative index returns are replaced with NaN.

If the calculation method is 'down', all zero and positive index returns are replaced with NaN.

If the calculation method is the normal mode (''), a copy of the input data frame is returned unchanged.


In [ ]:
def Step3(input_df: pd.DataFrame, beta: str = '') -> pd.DataFrame:
    step3 = pd.DataFrame()

    if beta.lower()=='up':        
        step3['Stock Return'] = input_df['Stock Return']
        step3['Only Positive Index Return'] = input_df.apply(lambda x: np.nan if x['Index Return'] <= 0 else x['Index Return'],axis=1)
        #step3 = step3.dropna()
    elif beta.lower()=='down':        
        step3['Stock Return'] = input_df['Stock Return']
        step3['Only Negative Index Return'] =  input_df.apply(lambda x: np.nan if x['Index Return'] >= 0 else x['Index Return'],axis=1)
        #temp = temp.dropna()
    else:
        return input_df.copy()
    
    return step3

 ###  Examples
The function can be called like this:

__Normal Beta__
```
Step3_df = Step3(Step2_df, '')
```
__Beta Up__
```
Step3_df = Step3(Step2_df, 'up')
```
__Beta Down__
```
Step3_df = Step3(Step2_df, 'down')
```

In [ ]:
Step3_df = Step3(Step2_df, 'up')
Step3_df

## Step 4: Series of negative or positive Index Return

The function defined for this step takes the data frame produced in the third step, removes all unavailable data, and returns the cleaned data frame to the caller.


In [ ]:
def Step4(input_df: pd.DataFrame) -> pd.DataFrame:
    return input_df.dropna()

###  Example
The function can be called like this:

```
Step4_df = Step4(Step3_df)
```

In [ ]:
Step4_df = Step4(Step3_df)
Step4_df

## Step 5: Calculation of BETA's value

The function defined for this step uses the data frame produced in the fourth step to calculate a beta value.

It applies the __numpy.polyfit__ function to estimate the relationship between stock returns and index returns—a common approach for computing beta in finance. The function passes the stock returns as the first argument (the x-coordinates of the sample points), the index returns as the second argument (the y-coordinates), and 1 as the third argument to indicate a first‑degree polynomial fit.

The function then returns the first value produced by __numpy.polyfit__, which corresponds to the slope of the fitted line.



In [ ]:
def Step5(input_df: pd.DataFrame):
    slope, _ = np.polyfit(
        x=input_df.iloc[:, 1].tolist(), 
        y=input_df.iloc[:,0].tolist(), 
        deg=1)    
    return slope

###  Example
The function can be called like this:

```
Beta_value = Step5(Step4_df)
```

In [ ]:
Beta_value = Step5(Step4_df)
Beta_value

## Usage
To use these functions, you must define a list of parameters used to calculate beta values. Each entry includes the following properties:

- __Name__: The name of Beta value
- __Periodicity__: An interval string for filtering historical pricing events, such as P1D (daily), P1W (weekly), and P1M (monthly)
- __Data Points__: A maximum number of rows to return
- __Rule of % Available__: A percentage number of data points to be available for calculation
- __Calculation Method__: The calculation method used to compute the beta value. Valid options include '', 'up', or 'down'

The following code defines 11 rules used to calculate beta values.

In [ ]:

beta_data = [
    {
        'Name':'Beta - 90 Days - Daily',
        'Periodcity':'P1D',
        'Data Points':91,
        'Rule of % Avaialble': 67,
        'Calculation Method': ''
    },
    {
        'Name':'Beta - 181 Days - Daily',
        'Periodcity':'P1D',
        'Data Points':181,
        'Rule of % Avaialble': 67,
        'Calculation Method': ''
    },
    {
        'Name':'Beta - 2 Years - Weekly',
        'Periodcity':'P1W',
        'Data Points':105,
        'Rule of % Avaialble': 67,
        'Calculation Method': ''
    },
    {
        'Name':'Beta - 3 Years - Weekly',
        'Periodcity':'P1W',
        'Data Points':157,
        'Rule of % Avaialble': 67,
        'Calculation Method': ''
    },
    {
        'Name':'Beta - 5 Years - Monthly',
        'Periodcity':'P1M',
        'Data Points':61,
        'Rule of % Avaialble': 67,
        'Calculation Method': ''
    },
    {
        'Name':'Beta Up - 2 Years - Weekly',
        'Periodcity':'P1W',
        'Data Points':105,
        'Rule of % Avaialble': 17,
        'Calculation Method': 'up'
    },
    {
        'Name':'Beta Up - 3 Years - Weekly',
        'Periodcity':'P1W',
        'Data Points':157,
        'Rule of % Avaialble': 17,
        'Calculation Method': 'up'
    },
    {
        'Name':'Beta Up - 5 Years - Monthly',
        'Periodcity':'P1M',
        'Data Points':61,
        'Rule of % Avaialble': 17,
        'Calculation Method': 'up'
    },
    {
        'Name':'Beta Down - 2 Years - Weekly',
        'Periodcity':'P1W',
        'Data Points':105,
        'Rule of % Avaialble': 17,
        'Calculation Method': 'down'
    },
    {
        'Name':'Beta Down - 3 Years - Weekly',
        'Periodcity':'P1W',
        'Data Points':157,
        'Rule of % Avaialble': 17,
        'Calculation Method': 'down'
    },
    {
        'Name':'Beta Down - 5 Years - Monthly',
        'Periodcity':'P1M',
        'Data Points':61,
        'Rule of % Avaialble': 17,
        'Calculation Method': 'down'
    }
]
beta_table = pd.DataFrame(beta_data)

beta_table
        

Next, define a function that iterates through all entries in the list and calculates the beta values for each one. The function accepts the following parameters:

- __input_df__ (pd.DataFrame): The data frame containing the rules for calculating beta values.
- __ric__ (str): The stock RIC.
- __index__ (str): The index RIC.
- __end__ (str): The end-date string for querying events, formatted as YYYY‑MM‑DD.

This function invokes the methods defined in Steps 1 through 5 and returns a data frame containing the resulting beta values.

In [ ]:
def CalculateBeta(input_df: pd.DataFrame, ric: str, index: str, end: str) -> pd.DataFrame:
    column_names = [
        'Name', 
        'Numbers of Stocks Returns', 
        'Numbers of Stocks Returns, relative to Beta Analysis',
        '% of data results',
        'Rule of % Available',
        'Results Analysis',
        'Values']
    summary_df = pd.DataFrame(columns=column_names)
    for i, row in input_df.iterrows():        
        step1_df = Step1(ric, index,  row['Periodcity'], end, row['Data Points'])
        #display(step1_df)
        step2_df = Step2(step1_df)
        #display(step2_df)
        step3_df = Step3(step2_df, row['Calculation Method'])
        #display(step3_df)
        step4_df = Step4(step3_df)
        #display(step4_df)
        step5 = Step5(step4_df)
        if row['Calculation Method'] == '':
            summary_df.loc[len(summary_df)]=[row['Name'],len(step2_df),None, None, row['Rule of % Avaialble'], None, step5]
        else:
            summary_df.loc[len(summary_df)]=[row['Name'],len(step2_df),len(step4_df), (len(step4_df) / len(step2_df))*100, row['Rule of % Avaialble'], 'Beta Valid' if  (len(step4_df) / len(step2_df))*100 >= row['Rule of % Avaialble'] else 'Beta Invalid' , step5]
    return summary_df
    

### Example
The function can be called like this:
```
ric = 'PTT.BK'
index = '.SETI'
end = '2026-01-23'
CalculateBeta(beta_table, ric, index, end)
```

In [ ]:
from datetime import datetime

ric = 'PTT.BK'
index = '.SETI'
end = datetime.today().strftime('%Y-%m-%d')
CalculateBeta(beta_table, ric, index, end)

## Summary

The article provides a step‑by‑step translation of the Workspace Excel formulas used in the __Company Beta Types – Historical__ Excel template into Python code. Its primary goal is to demonstrate how to compute three key beta measures—Normal Beta, Beta Up, and Beta Down—for any selected company and benchmark index on any specified date.

To achieve this, the article leverages the [LSEG Data Library for Python](https://developers.lseg.com/en/api-catalog/lseg-data-platform/lseg-data-library-for-python), which serves as the data‑access layer for retrieving historical closing prices for both the target stock and its corresponding index. Once the historical price series is obtained, the workflow proceeds by transforming the prices into return series, typically using daily percentage changes or log returns, depending on the analytical convention.

After constructing the return series for both the stock and the index, the article explains how to apply the same logic embedded in the Excel template to compute the different beta types. Normal Beta is derived from the covariance of stock and index returns relative to the variance of index returns. Beta Up and Beta Down are calculated using return subsets filtered on whether index returns are positive or negative, respectively—mirroring the conditional calculations performed in the Workspace Excel template.

By walking through each stage of the process—from data retrieval to return transformation to beta estimation—the article not only replicates the Excel workflow but also illustrates how these computations can be automated, reproduced, and scaled in Python for broader analytical applications.
